# ✉️ Handwritten Digit Recognition — The Post Office Problem
## A Complete CNN Tutorial with MNIST

---

### 🧠 The Famous Problem

In the late 1980s, the U.S. Postal Service faced a massive challenge: **millions of handwritten ZIP codes** needed to be sorted every day. Human workers were too slow and too expensive. Could a machine learn to read handwriting?

**Yann LeCun** and his colleagues at Bell Labs answered this question in 1989 by inventing **LeNet**, one of the first Convolutional Neural Networks (CNNs). It could read digits with near-human accuracy — a landmark moment in the history of AI.

The dataset they used — **MNIST** — became the "Hello World" of machine learning. It contains:
- **60,000** training images of handwritten digits (0–9)
- **10,000** test images
- Each image is **28×28 pixels**, grayscale

In this notebook, we will **build this system from scratch**, step by step, with full explanations.

---

### 📋 What You Will Learn

1. What the data looks like and how images are represented as numbers
2. How a simple Fully Connected Network (Dense) struggles with images
3. **Why CNNs are better** — the core intuition behind convolutions
4. Building a CNN layer by layer
5. Training the network and reading the results
6. Visualizing what the network "sees" inside
7. Testing on your own drawn digits!

---

> **No prior deep learning knowledge required.** Every concept is explained in plain language before the code.

## 📦 Step 0 — Install and Import Everything

We use:
- **TensorFlow / Keras** — to build and train neural networks
- **NumPy** — for numerical array operations
- **Matplotlib** — for visualizing images and results
- **Seaborn** — for pretty confusion matrices

In [ ]:
# Install dependencies (run once; comment out after)
# !pip install tensorflow numpy matplotlib seaborn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

# For reproducibility — same results every time you run
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print("✅ All imports successful!")

---
## 🖼️ Step 1 — Load and Explore the Data

### What is an image, mathematically?

A grayscale image is simply a **2D grid of numbers**. Each number (called a **pixel**) stores a brightness value:
- `0` = black
- `255` = white
- Anything in between = a shade of gray

A 28×28 MNIST image is a matrix of 784 numbers. The network must learn to interpret these 784 numbers and decide: *is this a 0, a 1, a 2 ... or a 9?*

In [ ]:
# Load the MNIST dataset — Keras downloads it automatically
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("=== Dataset Shapes ===")
print(f"Training images : {X_train.shape}  → {X_train.shape[0]} images, each {X_train.shape[1]}×{X_train.shape[2]} pixels")
print(f"Training labels : {y_train.shape}  → one label per image (0–9)")
print(f"Test images     : {X_test.shape}")
print(f"Test labels     : {y_test.shape}")

print(f"\nPixel value range: {X_train.min()} to {X_train.max()}")
print(f"Label values (unique digits): {np.unique(y_train)}")

In [ ]:
# -------------------------------------------------------
# Visualize sample images from the training set
# -------------------------------------------------------
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
fig.suptitle('Sample MNIST Images — Each column is one digit (0–9)', 
             fontsize=14, fontweight='bold', y=1.02)

for digit in range(10):
    # Find 3 examples of each digit
    indices = np.where(y_train == digit)[0][:3]
    for row, idx in enumerate(indices):
        ax = axes[row, digit]
        ax.imshow(X_train[idx], cmap='gray')
        ax.axis('off')
        if row == 0:
            ax.set_title(str(digit), fontsize=14, fontweight='bold', color='navy')

plt.tight_layout()
plt.show()

print("Notice: The same digit looks different across samples.")
print("People write differently — the network must handle this variation!")

In [ ]:
# -------------------------------------------------------
# Look at a SINGLE image as raw pixel numbers
# -------------------------------------------------------
example_idx = 0
example_image = X_train[example_idx]
example_label = y_train[example_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: the image visually
axes[0].imshow(example_image, cmap='gray')
axes[0].set_title(f'Image (label = {example_label})', fontsize=13)
axes[0].axis('off')

# Right: the raw pixel numbers
im = axes[1].imshow(example_image, cmap='Blues')
axes[1].set_title('Same image as pixel values (0=black, 255=white)', fontsize=13)
for i in range(28):
    for j in range(28):
        val = example_image[i, j]
        if val > 10:  # only annotate non-background pixels
            axes[1].text(j, i, str(val), ha='center', va='center',
                        fontsize=4, color='white' if val > 128 else 'black')

plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

print(f"This image contains the digit: {example_label}")
print(f"It is represented as a {example_image.shape} array of integers.")

In [ ]:
# -------------------------------------------------------
# Check class distribution — are all digits equally common?
# -------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (data, name) in zip(axes, [(y_train, 'Training'), (y_test, 'Test')]):
    unique, counts = np.unique(data, return_counts=True)
    bars = ax.bar(unique, counts, color=plt.cm.tab10.colors, edgecolor='black', linewidth=0.5)
    ax.set_title(f'{name} Set Class Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Digit')
    ax.set_ylabel('Count')
    ax.set_xticks(range(10))
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                str(count), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("✅ Good news: the dataset is fairly balanced.")
print("Each digit appears roughly 6,000 times in training and 1,000 times in testing.")

---
## 🔧 Step 2 — Preprocess the Data

### Why preprocess?

Neural networks work best when input values are **small numbers close to 0**. Raw pixels go from 0 to 255. We normalize them to the range **[0, 1]** by dividing by 255.

Think of it like this: if you're doing a math problem where numbers are in the millions vs. the thousands, the millions dominate and distort your sense of scale. Normalization puts everyone on equal footing.

### Reshaping for CNN
CNNs expect input of shape `(height, width, channels)`. Since our images are grayscale, channels = 1. We need to add that dimension.

In [ ]:
# -------------------------------------------------------
# Normalize pixel values from [0, 255] to [0.0, 1.0]
# -------------------------------------------------------
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm  = X_test.astype('float32')  / 255.0

print(f"Before normalization — min: {X_train.min()}, max: {X_train.max()}")
print(f"After normalization  — min: {X_train_norm.min():.2f}, max: {X_train_norm.max():.2f}")

# -------------------------------------------------------
# Reshape: add channel dimension  (28, 28) → (28, 28, 1)
# -------------------------------------------------------
# CNN input shape: (batch_size, height, width, channels)
X_train_cnn = X_train_norm.reshape(-1, 28, 28, 1)
X_test_cnn  = X_test_norm.reshape(-1, 28, 28, 1)

print(f"\nCNN input shape per image: {X_train_cnn.shape[1:]}")
print(f"Full training tensor shape: {X_train_cnn.shape}")

# -------------------------------------------------------
# One-hot encode labels
# -------------------------------------------------------
# Instead of label = 5, we want label = [0,0,0,0,0,1,0,0,0,0]
# This lets the output layer have one neuron per class
y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test, 10)

print(f"\nOriginal label example: {y_train[0]}")
print(f"One-hot label example : {y_train_cat[0]}")
print("  → position 5 is 1 because this is the digit '5'")

---
## 🏗️ Step 3 — Why Not Just Use a Flat Neural Network?

### The "Flatten First" Approach

A naive approach: flatten the 28×28 image into 784 numbers, feed them into a fully-connected network. Let's try it and see its limitations.

### The Problem with Flattening

When you flatten `[row0, row1, row2, ...]` into one long array, **spatial relationships are lost**. Pixel (row=5, col=3) is near pixel (row=5, col=4) — but after flattening, they might be 200 positions apart. The network has no way of knowing they were neighbors.

Also, a shifted or rotated image becomes completely different to a flat network, even though it's the same digit.

In [ ]:
# -------------------------------------------------------
# Build a simple Dense (Fully Connected) network
# -------------------------------------------------------

def build_dense_model():
    model = models.Sequential([
        # Step 1: Flatten the 28x28 image into 784 numbers
        layers.Flatten(input_shape=(28, 28, 1)),
        
        # Step 2: A hidden layer with 128 neurons
        # 'relu' activation: f(x) = max(0, x)  — kills negative signals
        layers.Dense(128, activation='relu'),
        
        # Step 3: Another hidden layer
        layers.Dense(64, activation='relu'),
        
        # Step 4: Output — 10 neurons, one per digit
        # 'softmax' converts raw scores to probabilities that sum to 1
        layers.Dense(10, activation='softmax')
    ], name='Dense_Baseline')
    return model

dense_model = build_dense_model()
dense_model.summary()

In [ ]:
# -------------------------------------------------------
# Compile and train the Dense model
# -------------------------------------------------------
# compile() defines:
#   optimizer  = how to update weights (Adam is a smart gradient descent)
#   loss       = what to minimize (cross-entropy for classification)
#   metrics    = what to report during training

dense_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Training Dense model...")
dense_history = dense_model.fit(
    X_train_cnn, y_train_cat,
    epochs=10,
    batch_size=128,       # Process 128 images at a time before updating weights
    validation_split=0.1, # Keep 10% of training data for validation
    verbose=1
)

In [ ]:
# Evaluate on the test set
dense_loss, dense_acc = dense_model.evaluate(X_test_cnn, y_test_cat, verbose=0)
print(f"Dense Network — Test Accuracy: {dense_acc*100:.2f}%")
print(f"Dense Network — Test Loss    : {dense_loss:.4f}")

---
## 🔬 Step 4 — Understanding Convolutions (The Key Idea)

### What is a Convolution?

A convolution is a **sliding window operation**. We take a small matrix (called a **filter** or **kernel**), slide it across the image, and at each position we compute a dot product.

The result is a **feature map** — a new image that highlights a specific pattern (edges, corners, curves, etc.).

```
Image patch:        Filter (edge detector):   Result:
[0,  0,  0]         [-1, -1, -1]             
[0, 255, 0]    ×    [ 0,  0,  0]    →   high value (edge found!)
[0, 255, 0]         [ 1,  1,  1]             
```

### Why is this better than Dense layers?

1. **Parameter sharing**: One filter is applied everywhere — fewer parameters to learn
2. **Translation invariance**: A "7 detector" filter works whether the 7 is in the top-left or bottom-right
3. **Spatial awareness**: The filter knows about neighboring pixels

Let's visualize this with real filters on a real image:

In [ ]:
# -------------------------------------------------------
# Demonstrate convolution with hand-crafted filters
# -------------------------------------------------------

from scipy.ndimage import convolve

# Pick a sample image
sample = X_train_norm[0]  # shape (28, 28)

# Define 4 classic image processing filters
filters = {
    'Original': None,
    'Horizontal Edge\nDetector': np.array([[-1,-1,-1],
                                           [ 0, 0, 0],
                                           [ 1, 1, 1]]),
    'Vertical Edge\nDetector':   np.array([[-1, 0, 1],
                                           [-1, 0, 1],
                                           [-1, 0, 1]]),
    'Blur (Smoothing)':          np.ones((3,3)) / 9,
    'Sharpen':                   np.array([[ 0,-1, 0],
                                           [-1, 5,-1],
                                           [ 0,-1, 0]]),
}

fig, axes = plt.subplots(1, len(filters), figsize=(16, 3))
fig.suptitle('What Different Filters "See" in the Same Image', 
             fontsize=13, fontweight='bold')

for ax, (name, filt) in zip(axes, filters.items()):
    if filt is None:
        ax.imshow(sample, cmap='gray')
    else:
        filtered = convolve(sample, filt)
        ax.imshow(filtered, cmap='gray')
    ax.set_title(name, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Key insight: A CNN LEARNS these filters from data!")
print("It doesn't need us to design them — it figures out the best ones automatically.")

In [ ]:
# -------------------------------------------------------
# Visualize the convolution operation step by step
# -------------------------------------------------------

# Show a small 8x8 patch and how a 3x3 filter slides over it
patch = X_train_norm[0, 8:16, 10:18]  # 8x8 region of the first image
kernel = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]])  # Sobel vertical edge filter

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('How a Convolution Works — Step by Step', fontsize=13, fontweight='bold')

# Image patch
im0 = axes[0].imshow(patch, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Input Patch (8×8)', fontsize=11)
for i in range(8):
    for j in range(8):
        axes[0].text(j, i, f'{patch[i,j]:.1f}', ha='center', va='center',
                    fontsize=7, color='red' if patch[i,j] > 0.5 else 'white')
axes[0].axis('off')

# Kernel
axes[1].imshow(kernel, cmap='RdBu', vmin=-2, vmax=2)
axes[1].set_title('Filter / Kernel (3×3)', fontsize=11)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, str(kernel[i,j]), ha='center', va='center',
                    fontsize=12, fontweight='bold', color='black')
axes[1].axis('off')

# Result
result = convolve(patch, kernel)
axes[2].imshow(result, cmap='gray')
axes[2].set_title('Output Feature Map (6×6 after convolution)', fontsize=11)
for i in range(result.shape[0]):
    for j in range(result.shape[1]):
        axes[2].text(j, i, f'{result[i,j]:.1f}', ha='center', va='center',
                    fontsize=6, color='red')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("At each position, the filter does: sum(patch_region × filter_values)")
print("This produces ONE number per filter position → that's one pixel in the feature map.")

---
## 🧱 Step 5 — Build the CNN Architecture

### Architecture Overview

Our CNN follows this flow:

```
Input (28×28×1)
    ↓
[Conv2D: 32 filters, 3×3]  →  detect simple features (edges, blobs)
[BatchNorm + ReLU]
[MaxPooling 2×2]            →  shrink: 28×28 → 14×14
    ↓
[Conv2D: 64 filters, 3×3]  →  detect complex features (curves, corners)
[BatchNorm + ReLU]
[MaxPooling 2×2]            →  shrink: 14×14 → 7×7
    ↓
[Conv2D: 128 filters, 3×3] →  detect high-level features (digit parts)
[BatchNorm + ReLU]
    ↓
[GlobalAveragePooling]      →  collapse spatial dimensions
[Dense 256, Dropout 0.5]   →  combine all features
[Dense 10, Softmax]         →  output probabilities for 0–9
```

### Key Layers Explained

| Layer | Purpose |
|-------|--------|
| **Conv2D** | Applies learned filters to detect features |
| **BatchNorm** | Normalizes layer outputs → faster, more stable training |
| **ReLU** | Non-linearity: `max(0, x)` — lets network learn complex patterns |
| **MaxPooling** | Takes the max in each 2×2 region → reduces size, keeps strongest signals |
| **Dropout** | Randomly disables neurons during training → prevents overfitting |
| **Dense** | Fully connected layer — combines all learned features |
| **Softmax** | Converts scores to probabilities (sum = 1) |

In [ ]:
# -------------------------------------------------------
# Build the CNN model
# -------------------------------------------------------

def build_cnn_model():
    model = models.Sequential(name='Digit_CNN')
    
    # ── BLOCK 1 ──────────────────────────────────────────
    # 32 filters, each 3×3, padding='same' keeps the 28×28 size
    model.add(layers.Conv2D(32, (3, 3), padding='same',
                            input_shape=(28, 28, 1)))
    model.add(layers.BatchNormalization())  # stabilize learning
    model.add(layers.Activation('relu'))    # introduce non-linearity
    model.add(layers.Conv2D(32, (3, 3), padding='same'))  # second conv
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))  # 28×28 → 14×14
    model.add(layers.Dropout(0.25))         # drop 25% of neurons randomly

    # ── BLOCK 2 ──────────────────────────────────────────
    # Deeper: 64 filters detect more complex patterns
    model.add(layers.Conv2D(64, (3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(64, (3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))  # 14×14 → 7×7
    model.add(layers.Dropout(0.25))

    # ── BLOCK 3 ──────────────────────────────────────────
    # Even deeper: 128 filters for high-level patterns
    model.add(layers.Conv2D(128, (3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Dropout(0.25))

    # ── CLASSIFIER HEAD ──────────────────────────────────
    # GlobalAveragePooling: average across each feature map → 128 values
    model.add(layers.GlobalAveragePooling2D())
    
    model.add(layers.Dense(256))            # combine all 128 feature averages
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Dropout(0.5))          # heavy dropout before output

    # Output: 10 neurons (one per digit), softmax for probabilities
    model.add(layers.Dense(10, activation='softmax'))
    
    return model

cnn_model = build_cnn_model()
cnn_model.summary()

In [ ]:
# -------------------------------------------------------
# Visualize architecture as a diagram
# -------------------------------------------------------

# Count parameters per layer type
total_params = cnn_model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in cnn_model.trainable_weights])

print("=== Architecture Summary ===")
print(f"Total parameters  : {total_params:,}")
print(f"Trainable params  : {trainable_params:,}")
print()

# Compare with dense model
dense_params = dense_model.count_params()
print(f"Dense model params: {dense_params:,}")
print(f"CNN model params  : {total_params:,}")
print()
print("Note: CNN has more params than this simple dense net, but uses them more wisely")
print("because filters are shared across all spatial positions.")

---
## 🚀 Step 6 — Train the CNN

### What Happens During Training?

Training is an iterative process:
1. **Forward pass**: Feed images through the network → get predictions
2. **Compute loss**: Compare predictions to true labels (how wrong are we?)
3. **Backward pass**: Use backpropagation to compute gradients (how to fix the error?)
4. **Update weights**: Adjust all filters and connections slightly to reduce error
5. Repeat for the next batch

### Key Training Tricks We Use
- **Learning Rate Scheduling**: Start with a bigger learning rate, reduce it over time
- **Early Stopping**: Stop training when validation accuracy stops improving
- **Data Augmentation**: Slightly shift/rotate images to make the model more robust

In [ ]:
# -------------------------------------------------------
# Data Augmentation — artificially increase dataset variety
# -------------------------------------------------------
# We slightly shift and rotate images during training.
# This forces the model to generalize — not memorize exact positions.

from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=10,       # rotate up to 10 degrees
    width_shift_range=0.1,   # shift horizontally up to 10%
    height_shift_range=0.1,  # shift vertically up to 10%
    zoom_range=0.1           # zoom in/out up to 10%
)
datagen.fit(X_train_cnn)

# Visualize augmented samples
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
fig.suptitle('Original (top) vs Augmented (bottom)', fontsize=12, fontweight='bold')

for i in range(10):
    axes[0, i].imshow(X_train_cnn[i, :, :, 0], cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title(f'{y_train[i]}', fontsize=11, color='navy')

# Generate one batch of augmented images
aug_batch = next(datagen.flow(X_train_cnn[:10], batch_size=10, shuffle=False))
for i in range(10):
    axes[1, i].imshow(aug_batch[i, :, :, 0], cmap='gray')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print("Augmented images look slightly shifted/rotated but are still recognizable.")
print("This teaches the network to be position-invariant.")

In [ ]:
# -------------------------------------------------------
# Compile the CNN
# -------------------------------------------------------

cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# -------------------------------------------------------
# Define callbacks (smart training helpers)
# -------------------------------------------------------

callbacks = [
    # Reduce learning rate when validation accuracy plateaus
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,       # halve the LR
        patience=3,       # wait 3 epochs before reducing
        min_lr=1e-6,
        verbose=1
    ),
    # Stop training early if no improvement for 7 epochs
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        restore_best_weights=True,  # keep the best weights found
        verbose=1
    )
]

print("Model compiled. Ready to train!")

In [ ]:
# -------------------------------------------------------
# TRAIN! (This is the main event)
# -------------------------------------------------------

BATCH_SIZE = 128
EPOCHS     = 30   # max epochs; early stopping may end it sooner

steps_per_epoch = len(X_train_cnn) // BATCH_SIZE

print(f"Training on {len(X_train_cnn)} images, batch size {BATCH_SIZE}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Max epochs: {EPOCHS}\n")

history = cnn_model.fit(
    datagen.flow(X_train_cnn, y_train_cat, batch_size=BATCH_SIZE),
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=(X_test_cnn, y_test_cat),
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete!")

---
## 📊 Step 7 — Evaluate Performance

Now let's see how well our network learned. We'll look at:
1. Training curves (accuracy and loss over time)
2. Final test accuracy
3. Confusion matrix — which digits get confused with which?
4. Wrong predictions — can we spot the hard cases?

In [ ]:
# -------------------------------------------------------
# Plot training history
# -------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CNN Training History', fontsize=14, fontweight='bold')

epochs_range = range(1, len(history.history['accuracy']) + 1)

# Accuracy plot
axes[0].plot(epochs_range, history.history['accuracy'],     'b-o', label='Train', markersize=4)
axes[0].plot(epochs_range, history.history['val_accuracy'], 'r-o', label='Validation', markersize=4)
axes[0].set_title('Accuracy over Epochs', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].set_ylim([0.9, 1.0])
axes[0].grid(True, alpha=0.3)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.1f}%'))

# Loss plot
axes[1].plot(epochs_range, history.history['loss'],     'b-o', label='Train', markersize=4)
axes[1].plot(epochs_range, history.history['val_loss'], 'r-o', label='Validation', markersize=4)
axes[1].set_title('Loss over Epochs', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("What to look for:")
print("  • Accuracy should go UP over epochs")
print("  • Loss should go DOWN over epochs")
print("  • Train and Validation lines close together = good (no overfitting)")
print("  • Big gap between them = overfitting (model memorized training data)")

In [ ]:
# -------------------------------------------------------
# Final test accuracy — the moment of truth
# -------------------------------------------------------

cnn_loss, cnn_acc = cnn_model.evaluate(X_test_cnn, y_test_cat, verbose=0)

print("=" * 50)
print("         MODEL COMPARISON")
print("=" * 50)
print(f"Dense Network : {dense_acc*100:.2f}% accuracy")
print(f"CNN           : {cnn_acc*100:.2f}% accuracy")
print(f"Improvement   : +{(cnn_acc - dense_acc)*100:.2f} percentage points")
print("=" * 50)
print()
print(f"At {cnn_acc*100:.2f}% accuracy, the CNN makes ~{int((1-cnn_acc)*10000)} errors on 10,000 test images.")
print("Human-level performance on MNIST is around 98–99%.")
print("State-of-the-art models reach 99.7%+ with capsule networks or ensembles.")

In [ ]:
# -------------------------------------------------------
# Confusion Matrix — where does the model go wrong?
# -------------------------------------------------------

y_pred_probs = cnn_model.predict(X_test_cnn, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)  # take the highest-probability class

cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10),
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Confusion Matrix (counts)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label', fontsize=12)

# Normalized (row percentages)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10),
            ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (normalized)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label', fontsize=12)

plt.tight_layout()
plt.show()

print("How to read the confusion matrix:")
print("  • Diagonal = correct predictions (we want these high!)")
print("  • Off-diagonal = mistakes (e.g., row=4, col=9 means '4' predicted as '9')")
print("  • Common confusion pairs: 4↔9, 3↔5, 7↔1")

In [ ]:
# -------------------------------------------------------
# Per-class precision, recall, F1 score
# -------------------------------------------------------

print("Per-Class Performance Report:")
print("-" * 60)
print(classification_report(y_test, y_pred, 
                            target_names=[str(i) for i in range(10)]))
print()
print("Precision: of all images predicted as '4', what % were actually '4'?")
print("Recall   : of all actual '4's, what % did we correctly identify?")
print("F1       : harmonic mean of precision and recall (0=worst, 1=best)")

In [ ]:
# -------------------------------------------------------
# Look at WRONG predictions in detail
# -------------------------------------------------------

errors = np.where(y_pred != y_test)[0]
print(f"Total errors: {len(errors)} out of {len(y_test)} ({len(errors)/len(y_test)*100:.2f}%)")

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle(f'Wrong Predictions — True vs Predicted (showing first {4*8} errors)',
             fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(errors):
        idx = errors[i]
        ax.imshow(X_test[idx], cmap='gray')
        true_label = y_test[idx]
        pred_label = y_pred[idx]
        confidence = y_pred_probs[idx][pred_label] * 100
        ax.set_title(f'T:{true_label} P:{pred_label}\n{confidence:.0f}%',
                    fontsize=8, color='red')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("T = True label, P = Predicted label, % = model confidence")
print("Notice: many of these are genuinely hard! Even humans might struggle.")

---
## 🔭 Step 8 — Look Inside the Network

### What Does the CNN Actually See?

We can extract the **intermediate feature maps** — what each convolutional layer outputs for a given input. This reveals what the network is paying attention to at each stage.

- **Early layers**: detect simple things (edges, blobs)
- **Middle layers**: detect complex shapes (curves, corners)
- **Late layers**: detect semantic features (parts of digits)

In [ ]:
# -------------------------------------------------------
# Extract and visualize feature maps (activations)
# -------------------------------------------------------

# Build a model that outputs intermediate layers
layer_names = [layer.name for layer in cnn_model.layers 
               if 'conv' in layer.name or 'activation' in layer.name]

# Create a sub-model that outputs the first conv layer's activations
first_conv_output = cnn_model.get_layer(layer_names[0]).output
activation_model_1 = keras.Model(inputs=cnn_model.input, outputs=first_conv_output)

# Pick a sample image
sample_idx = 7
sample_img = X_test_cnn[sample_idx:sample_idx+1]  # shape (1, 28, 28, 1)
true_label = y_test[sample_idx]

# Get the feature maps from first conv layer
feature_maps_1 = activation_model_1.predict(sample_img, verbose=0)

n_filters = min(32, feature_maps_1.shape[-1])

fig = plt.figure(figsize=(16, 10))
fig.suptitle(f'Feature Maps from First Conv Layer\n(Input digit: {true_label})',
             fontsize=14, fontweight='bold')

# Show original image
ax = fig.add_subplot(5, 8, 1)
ax.imshow(X_test[sample_idx], cmap='gray')
ax.set_title('Input', fontsize=9, fontweight='bold', color='red')
ax.axis('off')

# Show each filter's output
for i in range(min(n_filters, 39)):  # show up to 39 filter outputs
    ax = fig.add_subplot(5, 8, i + 2)
    ax.imshow(feature_maps_1[0, :, :, i], cmap='viridis')
    ax.set_title(f'F{i+1}', fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Each panel = what one filter detects in the input image.")
print("Bright areas = where that filter found its pattern.")
print("Different filters detect different features (edges, curves, etc.)")

In [ ]:
# -------------------------------------------------------
# Compare feature maps at different depths
# -------------------------------------------------------

# Get outputs from multiple layers
layer_outputs = []
layer_display_names = []

for layer in cnn_model.layers:
    if 'conv2d' in layer.name:
        layer_outputs.append(layer.output)
        layer_display_names.append(layer.name)

multi_model = keras.Model(inputs=cnn_model.input, outputs=layer_outputs)
all_feature_maps = multi_model.predict(sample_img, verbose=0)

# Plot 8 feature maps from each conv layer
n_layers = len(layer_display_names)
fig, axes = plt.subplots(n_layers, 9, figsize=(16, 2.5 * n_layers))
fig.suptitle('How Features Evolve Through the Network Layers', 
             fontsize=14, fontweight='bold')

if n_layers == 1:
    axes = [axes]

for layer_idx, (fmaps, name) in enumerate(zip(all_feature_maps, layer_display_names)):
    # Show input image in first column
    axes[layer_idx][0].imshow(X_test[sample_idx], cmap='gray')
    axes[layer_idx][0].set_ylabel(f'Layer {layer_idx+1}\n{name}', fontsize=8)
    axes[layer_idx][0].axis('off')
    
    # Show 8 feature maps
    for fi in range(8):
        ax = axes[layer_idx][fi+1]
        if fi < fmaps.shape[-1]:
            ax.imshow(fmaps[0, :, :, fi], cmap='plasma')
        ax.axis('off')

plt.tight_layout()
plt.show()

print("Notice:")
print("  • Early layers: detailed, recognizable patterns (looks like the digit)")
print("  • Later layers: abstract, smaller feature maps (the network has compressed info)")
print("  • This hierarchical feature extraction is CNN's superpower!")

In [ ]:
# -------------------------------------------------------
# Visualize learned filter weights
# -------------------------------------------------------

# Extract the actual learned weights from the first conv layer
first_layer = cnn_model.get_layer(layer_display_names[0])
weights, biases = first_layer.get_weights()

print(f"First conv layer filter shape: {weights.shape}")
print(f"  → {weights.shape[3]} filters, each {weights.shape[0]}×{weights.shape[1]}, for {weights.shape[2]} input channel")

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
fig.suptitle('Learned Filter Weights (First Conv Layer) — 32 filters of size 3×3',
             fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < weights.shape[3]:
        filt = weights[:, :, 0, i]  # 3×3 filter
        im = ax.imshow(filt, cmap='RdBu', vmin=-filt.max(), vmax=filt.max())
        ax.set_title(f'F{i+1}', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Blue = positive weight (activates on that pixel pattern)")
print("Red  = negative weight (suppresses on that pixel pattern)")
print("These filters were NOT designed by humans — the network learned them from data!")

---
## 🎯 Step 9 — Make Predictions with Confidence

Let's look at how the network outputs probability distributions, and examine both confident and uncertain predictions.

In [ ]:
# -------------------------------------------------------
# Show probability bars for individual predictions
# -------------------------------------------------------

def show_prediction(ax_img, ax_bar, image, true_label, pred_probs):
    """Display one image alongside its prediction probability bars."""
    pred_label = np.argmax(pred_probs)
    
    # Image
    ax_img.imshow(image, cmap='gray')
    color = 'green' if pred_label == true_label else 'red'
    ax_img.set_title(f'True: {true_label}\nPred: {pred_label}',
                    color=color, fontsize=10, fontweight='bold')
    ax_img.axis('off')
    
    # Probability bars
    colors = ['green' if i == true_label else
              'red'   if i == pred_label and pred_label != true_label else
              'steelblue' for i in range(10)]
    bars = ax_bar.bar(range(10), pred_probs, color=colors, edgecolor='black', linewidth=0.5)
    ax_bar.set_xticks(range(10))
    ax_bar.set_xlim(-0.5, 9.5)
    ax_bar.set_ylim(0, 1)
    ax_bar.set_xlabel('Digit')
    ax_bar.set_ylabel('Prob')
    ax_bar.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax_bar.grid(True, alpha=0.3, axis='y')

# Find 4 confident correct, 2 uncertain correct, 2 wrong predictions
correct_indices   = np.where(y_pred == y_test)[0]
incorrect_indices = np.where(y_pred != y_test)[0]

conf_correct = sorted(correct_indices, 
                      key=lambda i: -y_pred_probs[i, y_pred[i]])[:4]  # most confident
uncertain    = sorted(correct_indices,
                      key=lambda i: y_pred_probs[i, y_pred[i]])[:2]   # least confident (but correct!)
wrong        = incorrect_indices[:2]

selected = conf_correct + uncertain + wrong
labels   = ['Confident ✅']*4 + ['Uncertain ✅']*2 + ['Wrong ❌']*2

fig = plt.figure(figsize=(18, 12))
fig.suptitle('Model Predictions: Confident, Uncertain, and Wrong', 
             fontsize=14, fontweight='bold')

for i, (idx, label) in enumerate(zip(selected, labels)):
    ax_img = fig.add_subplot(4, 8, i*2 + 1 + (i // 4) * 8 - (i // 4) * 8)
    ax_bar = fig.add_subplot(4, 8, i*2 + 2 + (i // 4) * 8 - (i // 4) * 8)
    
    # Simpler layout
    row = i // 4
    col_pair = i % 4
    ax_img = fig.add_subplot(4, 8, row*8 + col_pair*2 + 1)
    ax_bar = fig.add_subplot(4, 8, row*8 + col_pair*2 + 2)
    show_prediction(ax_img, ax_bar, X_test[idx], y_test[idx], y_pred_probs[idx])
    if col_pair == 0:
        ax_img.set_ylabel(label, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 🎨 Step 10 — Test on Your Own Drawn Digit

This section lets you **draw a digit** using matplotlib's interactive canvas and get the CNN's prediction. (Works best in Jupyter Lab or classic Notebook with the widget backend.)

In [ ]:
# -------------------------------------------------------
# Interactive drawing tool (canvas)
# -------------------------------------------------------

from matplotlib.widgets import Button
from scipy.ndimage import center_of_mass, shift

# Canvas: 280x280 for easier drawing, will be resized to 28x28
CANVAS_SIZE = 280
canvas = np.zeros((CANVAS_SIZE, CANVAS_SIZE), dtype=np.float32)
drawing = [False]
last_pos = [None]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
plt.subplots_adjust(bottom=0.2)

ax_canvas = axes[0]
ax_pred   = axes[1]

ax_canvas.set_xlim(0, CANVAS_SIZE)
ax_canvas.set_ylim(0, CANVAS_SIZE)
ax_canvas.set_aspect('equal')
ax_canvas.set_facecolor('black')
ax_canvas.set_title('Draw a digit here! (Left click + drag)', fontsize=12)
img_display = ax_canvas.imshow(canvas, cmap='gray', vmin=0, vmax=1,
                               extent=[0, CANVAS_SIZE, 0, CANVAS_SIZE],
                               origin='upper')

bars = ax_pred.bar(range(10), np.zeros(10), color='steelblue', edgecolor='black')
ax_pred.set_xticks(range(10))
ax_pred.set_ylim(0, 1)
ax_pred.set_xlabel('Digit', fontsize=12)
ax_pred.set_ylabel('Probability', fontsize=12)
ax_pred.set_title('CNN Prediction', fontsize=12)
ax_pred.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
pred_text = ax_pred.text(0.5, 0.95, 'Draw something!', transform=ax_pred.transAxes,
                         ha='center', va='top', fontsize=14, fontweight='bold', color='navy')

def draw_circle(x, y, radius=12):
    """Draw a white circle on the canvas."""
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if dx**2 + dy**2 <= radius**2:
                px = int(y) + dy  # row
                py = int(x) + dx  # col
                if 0 <= px < CANVAS_SIZE and 0 <= py < CANVAS_SIZE:
                    canvas[px, py] = min(1.0, canvas[px, py] + 0.8)

def predict_canvas():
    """Resize canvas to 28x28 and run inference."""
    from PIL import Image
    # Convert to PIL, resize to 28x28
    pil_img = Image.fromarray((canvas * 255).astype(np.uint8))
    pil_small = pil_img.resize((28, 28), Image.LANCZOS)
    arr = np.array(pil_small) / 255.0
    arr = arr.reshape(1, 28, 28, 1)
    probs = cnn_model.predict(arr, verbose=0)[0]
    return probs

def on_press(event):
    if event.inaxes == ax_canvas and event.button == 1:
        drawing[0] = True
        draw_circle(event.xdata, CANVAS_SIZE - event.ydata)
        last_pos[0] = (event.xdata, CANVAS_SIZE - event.ydata)
        img_display.set_data(canvas)
        fig.canvas.draw_idle()

def on_release(event):
    if event.button == 1:
        drawing[0] = False
        last_pos[0] = None
        # Run prediction on mouse release
        try:
            probs = predict_canvas()
            pred = np.argmax(probs)
            for bar, p in zip(bars, probs):
                bar.set_height(p)
                bar.set_color('green' if np.argmax(probs) == bars.index(bar) else 'steelblue')
            for i, (bar, p) in enumerate(zip(bars, probs)):
                bar.set_height(p)
                bar.set_color('green' if i == pred else 'steelblue')
            pred_text.set_text(f'Predicted: {pred} ({probs[pred]*100:.1f}%)')
            ax_pred.set_title(f'CNN thinks this is: {pred}', fontsize=14, fontweight='bold', color='green')
            fig.canvas.draw_idle()
        except Exception as e:
            print(f"Prediction error: {e}")

def on_move(event):
    if drawing[0] and event.inaxes == ax_canvas and event.xdata and event.ydata:
        x, y = event.xdata, CANVAS_SIZE - event.ydata
        draw_circle(x, y)
        last_pos[0] = (x, y)
        img_display.set_data(canvas)
        fig.canvas.draw_idle()

# Clear button
ax_clear = plt.axes([0.2, 0.05, 0.15, 0.06])
btn_clear = Button(ax_clear, 'Clear Canvas', color='lightcoral', hovercolor='red')

def on_clear(event):
    canvas[:] = 0
    img_display.set_data(canvas)
    for bar in bars:
        bar.set_height(0)
    pred_text.set_text('Draw something!')
    ax_pred.set_title('CNN Prediction', fontsize=12)
    fig.canvas.draw_idle()

btn_clear.on_clicked(on_clear)
fig.canvas.mpl_connect('button_press_event', on_press)
fig.canvas.mpl_connect('button_release_event', on_release)
fig.canvas.mpl_connect('motion_notify_event', on_move)

plt.show()
print("Draw a digit with your mouse and release to see the prediction!")
print("Note: This works best in Jupyter Lab with %matplotlib widget enabled.")

---
## 🧪 Step 11 — Experiment: What Makes Digits Hard?

Let's dig into the most-confused pairs and understand *why*.

In [ ]:
# -------------------------------------------------------
# Find the most commonly confused digit pairs
# -------------------------------------------------------

# Get all errors and sort by frequency
error_pairs = [(y_test[i], y_pred[i]) for i in errors]
from collections import Counter
pair_counts = Counter(error_pairs).most_common(10)

print("Top 10 Most Common Confusion Pairs:")
print(f"{'True':>6} {'Predicted':>10} {'Count':>7}")
print("-" * 28)
for (true, pred), count in pair_counts:
    print(f"  {true:>4}  →  {pred:>4}       {count:>5}")

In [ ]:
# -------------------------------------------------------
# Show examples of each confused pair side-by-side
# -------------------------------------------------------

top_pairs = pair_counts[:6]

fig, axes = plt.subplots(6, 8, figsize=(16, 12))
fig.suptitle('Most Common Confusion Pairs — Can You See Why?', 
             fontsize=14, fontweight='bold')

for row, ((true_d, pred_d), count) in enumerate(top_pairs):
    # Find cases where true=true_d and pred=pred_d
    pair_indices = [i for i in errors 
                   if y_test[i] == true_d and y_pred[i] == pred_d]
    
    axes[row, 0].set_ylabel(f'True={true_d}\nPred={pred_d}\n(n={count})', 
                           fontsize=9, rotation=0, labelpad=60, va='center')
    
    for col in range(8):
        ax = axes[row, col]
        if col < len(pair_indices):
            idx = pair_indices[col]
            ax.imshow(X_test[idx], cmap='gray')
            conf = y_pred_probs[idx][pred_d] * 100
            ax.set_title(f'{conf:.0f}%', fontsize=8, color='red')
        ax.axis('off')

plt.tight_layout()
plt.show()

print("Many errors are ambiguous even to humans! This is the irreducible error floor.")

---
## 💾 Step 12 — Save and Load the Model

In a real post office system, you'd train once and then deploy the model to run 24/7. Keras makes saving and loading trivial.

In [ ]:
# -------------------------------------------------------
# Save the trained model
# -------------------------------------------------------

model_path = 'mnist_cnn_model.keras'
cnn_model.save(model_path)
print(f"Model saved to: {model_path}")

# -------------------------------------------------------
# Load the model back
# -------------------------------------------------------

loaded_model = keras.models.load_model(model_path)
print(f"Model loaded!")

# Verify it still works
_, loaded_acc = loaded_model.evaluate(X_test_cnn, y_test_cat, verbose=0)
print(f"Loaded model accuracy: {loaded_acc*100:.2f}%")
print("✅ Model saved and loaded successfully!")

---
## 🏆 Step 13 — Final Summary

### What We Built

We built a complete **handwritten digit recognition system** — the exact type of system used by post offices to read ZIP codes automatically.

### Key Results

In [ ]:
# -------------------------------------------------------
# Final summary comparison
# -------------------------------------------------------

print("=" * 55)
print("         FINAL RESULTS SUMMARY")
print("=" * 55)
print()
print(f"  Dense Neural Network   : {dense_acc*100:.2f}% accuracy")
print(f"  Our CNN                : {cnn_acc*100:.2f}% accuracy")
print(f"  Human-level estimate   : ~98-99% accuracy")
print(f"  State-of-the-art CNN   : ~99.7% accuracy")
print()
print(f"  Total CNN errors: {len(errors)} out of 10,000 test images")
print()
print("=" * 55)
print("         KEY LESSONS LEARNED")
print("=" * 55)
print()
print("  1. Images are just grids of numbers (0–255 per pixel)")
print("  2. CNNs use sliding filters to detect spatial patterns")
print("  3. Deep = hierarchical: edges → shapes → digits")
print("  4. Normalization and augmentation improve training")
print("  5. Dropout prevents overfitting (memorizing training data)")
print("  6. CNNs dramatically outperform flat Dense networks")
print("  7. Some errors are inherently ambiguous — even for humans")
print()
print("  This is exactly the architecture that solved")
print("  the Post Office problem in the 1990s!")
print("=" * 55)

In [ ]:
# -------------------------------------------------------
# Visual final summary
# -------------------------------------------------------

fig = plt.figure(figsize=(16, 9))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('CNN for Handwritten Digit Recognition — Complete Pipeline', 
             fontsize=16, fontweight='bold', color='white', y=0.98)

# Show some correct predictions in a strip
sample_indices = [np.where(y_test == d)[0][0] for d in range(10)]
images_row = [X_test[i] for i in sample_indices]
probs_row  = [y_pred_probs[i] for i in sample_indices]
preds_row  = [y_pred[i] for i in sample_indices]

for col, (img, probs, pred, true) in enumerate(zip(images_row, probs_row, preds_row,
                                                    [y_test[i] for i in sample_indices])):
    # Image
    ax_img = fig.add_subplot(3, 10, col + 1)
    ax_img.imshow(img, cmap='gray')
    ax_img.axis('off')
    ax_img.set_title(str(true), color='cyan', fontsize=14, fontweight='bold')
    
    # Arrow (text)
    ax_arr = fig.add_subplot(3, 10, col + 11)
    ax_arr.text(0.5, 0.5, '↓\nCNN', ha='center', va='center',
               color='yellow', fontsize=12, fontweight='bold')
    ax_arr.set_facecolor('#1a1a2e')
    ax_arr.axis('off')
    
    # Probability bar
    ax_bar = fig.add_subplot(3, 10, col + 21)
    colors = ['#00ff88' if i == pred else '#4444aa' for i in range(10)]
    ax_bar.bar(range(10), probs, color=colors, edgecolor='none')
    ax_bar.set_xticks(range(10))
    ax_bar.set_xticklabels(range(10), color='white', fontsize=7)
    ax_bar.set_ylim(0, 1)
    ax_bar.set_facecolor('#0f0f23')
    ax_bar.tick_params(colors='white')
    for spine in ax_bar.spines.values():
        spine.set_color('#333366')
    ax_bar.set_title(f'✓ {pred}  ({probs[pred]*100:.0f}%)', 
                    color='#00ff88', fontsize=9, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print("\n🎉 Congratulations! You've built a complete handwritten digit recognition system!")
print("This is the exact technology powering postal ZIP code readers worldwide.")

---
## 🚀 Going Further — Challenges for You

Now that you understand the basics, here are ways to go deeper:

### Immediate Experiments
1. **Change the number of filters** in each Conv layer — how does accuracy change?
2. **Remove Dropout** — does the model overfit?
3. **Remove Batch Normalization** — does training become unstable?
4. **Try without data augmentation** — does accuracy drop?

### Harder Challenges
5. **Fashion MNIST** — same format, but 10 clothing categories instead of digits. Try it with `keras.datasets.fashion_mnist`.
6. **CIFAR-10** — 32×32 color images (10 classes). Much harder!
7. **Add a 4th conv block** — can you squeeze out 0.1% more accuracy?
8. **Implement LeNet-5** — the original 1989 architecture by Yann LeCun
9. **Try transfer learning** — use a pretrained VGG16 on MNIST (educational overkill but instructive!)

### Real-World Extensions
10. **Multi-digit recognition** — segment a phone number image and read each digit
11. **CAPTCHA breaker** — apply this to simple text CAPTCHAs
12. **License plate recognition** — a real-world version of this problem!

---

### 📚 Key Papers and Resources

- **LeCun et al. (1989)**: "Backpropagation Applied to Handwritten Zip Code Recognition" — the original paper!
- **LeCun et al. (1998)**: "Gradient-Based Learning Applied to Document Recognition" — introduces LeNet-5
- **THE MNIST DATABASE**: http://yann.lecun.com/exdb/mnist/
- **CS231n** (Stanford): https://cs231n.github.io — the definitive CNN course

---

> *"The digital world runs on pattern recognition. You've just built a small but complete piece of that world."*